# Forecast California Housing Prices

In [20]:
# Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# Dataset
from sklearn.datasets import fetch_california_housing

In [21]:
# Set random seed
np.random.seed(50)
torch.manual_seed(50)

In [22]:
# Load the dataset
data = fetch_california_housing()
features = pd.DataFrame(data.data, columns=data.feature_names)
targets = pd.Series(data.target, name="Price").values.astype(float).reshape(-1, 1)

California Housing Dataset Information

A Bunch is a dictionary-like object used by scikit-learn to store datasets. 
It contains dataset information such as the data, target values, feature names, 
target names, and descriptions.

The default type for the dataset data is a NumPy array (numpy.ndarray).

The default type for the target values is also a NumPy array (numpy.ndarray).

In [23]:
# Split data into training and testing sets before scaling

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    features, targets, test_size=0.33, random_state=50
)

# Create scalers
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

# Fit scalers only on training data
X_train_scaled = feature_scaler.fit_transform(X_train_raw)
X_test_scaled = feature_scaler.transform(X_test_raw)

y_train_scaled = target_scaler.fit_transform(y_train_raw)
y_test_scaled = target_scaler.transform(y_test_raw)

In [24]:
def create_sequences(data, targets, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(targets[i + seq_length])
    return np.array(X), np.array(y)


seq_length = 20

# Data cleaning and preprocessing

# LSTM model

# GRU model

**Note:** The California housing dataset was sourced by scikit-learn from the StatLib repository: https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

**Reference**  
Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. Statistics & Probability Letters, 33(3), 291-297.

In [25]:
# Create sequences for training and testing data

X_train, y_train = create_sequences(X_train_scaled, y_train_scaled, seq_length)

X_test, y_test = create_sequences(X_test_scaled, y_test_scaled, seq_length)


# Convert to tensors

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [26]:
# create LSTM MODEL CLASS
class LSTMModel(nn.Module):

    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        output, _ = self.lstm(x)

        output = self.fc(output[:, -1, :])

        return output

In [27]:
# define hyperparameters
input_size = 8
hidden_size = 50
output_size = 1
num_layers = 2

learning_rate = 0.001
num_epochs = 100

# L2 regularisation strength
weight_decay = 0.01

In [28]:
# initialise model and loss

model = LSTMModel(input_size, hidden_size, output_size, num_layers)

criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [29]:
# training loop
for epoch in range(num_epochs):

    optimizer.zero_grad()

    output = model(X_train_tensor)

    loss = criterion(output, y_train_tensor)

    loss.backward()

    optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [1/100], Loss: 0.1687
Epoch [2/100], Loss: 0.1564
Epoch [3/100], Loss: 0.1447
Epoch [4/100], Loss: 0.1337
Epoch [5/100], Loss: 0.1233
Epoch [6/100], Loss: 0.1136
Epoch [7/100], Loss: 0.1044
Epoch [8/100], Loss: 0.0959
Epoch [9/100], Loss: 0.0881
Epoch [10/100], Loss: 0.0810
Epoch [11/100], Loss: 0.0747
Epoch [12/100], Loss: 0.0693
Epoch [13/100], Loss: 0.0648
Epoch [14/100], Loss: 0.0613
Epoch [15/100], Loss: 0.0589
Epoch [16/100], Loss: 0.0574
Epoch [17/100], Loss: 0.0568
Epoch [18/100], Loss: 0.0569
Epoch [19/100], Loss: 0.0573
Epoch [20/100], Loss: 0.0579
Epoch [21/100], Loss: 0.0584
Epoch [22/100], Loss: 0.0587
Epoch [23/100], Loss: 0.0588
Epoch [24/100], Loss: 0.0587
Epoch [25/100], Loss: 0.0584
Epoch [26/100], Loss: 0.0579
Epoch [27/100], Loss: 0.0575
Epoch [28/100], Loss: 0.0572
Epoch [29/100], Loss: 0.0569
Epoch [30/100], Loss: 0.0568
Epoch [31/100], Loss: 0.0568
Epoch [32/100], Loss: 0.0569
Epoch [33/100], Loss: 0.0570
Epoch [34/100], Loss: 0.0572
Epoch [35/100], Loss: 0

In [30]:
# predictions
model.eval()

with torch.no_grad():

    y_pred = model(X_test_tensor)


# Convert predictions back to original price scale

y_pred_original = target_scaler.inverse_transform(y_pred.cpu().numpy())

y_test_original = target_scaler.inverse_transform(y_test_tensor.cpu().numpy())


print(y_pred_original[:10])

[[2.020513 ]
 [2.0222225]
 [2.0252726]
 [2.0202384]
 [2.0121593]
 [2.0037873]
 [1.999334 ]
 [1.9985309]
 [1.9954683]
 [1.9961052]]


In [31]:
# calculate
mse = mean_squared_error(y_test_original, y_pred_original)

print("Mean Squared Error:", mse)

Mean Squared Error: 1.3247904777526855


Model Performance

The final model achieved a Mean Squared Error (MSE) of approximately 1.32 on the test set.

This indicates that the model's predictions were relatively close to the actual target values, with lower MSE values representing better predictive performance. Although some prediction error remains, the model was able to learn meaningful patterns from the sequential housing data.

PRACTICAL TASK TWO

In [32]:
# Convert to tensors

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [33]:
# Create dataloaders

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create loaders
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [34]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)

        # Take the output from the last time step
        out = self.fc(out[:, -1, :])

        return out

In [ ]:
input_size = X_train.shape[2]
hidden_size = 64
num_layers = 2
output_size = 1
learning_rate = 0.001
epochs = 50
batch_size = 64
lstm_model = LSTMModel(input_size, hidden_size, num_layers, output_size)

In [36]:
# loss function and optimizer with L2 regularisation

criterion = nn.MSELoss()

weight_decay = 0.01

optimizer = torch.optim.Adam(
    lstm_model.parameters(), lr=learning_rate, weight_decay=weight_decay
)

In [37]:
print("X_train_tensor shape:", X_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("Batch size:", train_loader.batch_size)
print("Number of batches:", len(train_loader))

X_train_tensor shape: torch.Size([13808, 20, 8])
y_train_tensor shape: torch.Size([13808, 1])
Batch size: 64
Number of batches: 216


In [38]:
for epoch in range(epochs):

    lstm_model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = lstm_model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/5, Loss: 0.0596
Epoch 2/5, Loss: 0.0569
Epoch 3/5, Loss: 0.0570
Epoch 4/5, Loss: 0.0568
Epoch 5/5, Loss: 0.0569


In [39]:
lstm_model.eval()

with torch.no_grad():

    predictions = lstm_model(X_test_tensor)


# Convert predictions and actual values back to original scale

predictions_original = target_scaler.inverse_transform(predictions.numpy())

y_test_original = target_scaler.inverse_transform(y_test_tensor.numpy())


# Calculate MSE on original housing price scale

lstm_mse = mean_squared_error(y_test_original, predictions_original)

print("LSTM Test MSE:", lstm_mse)

LSTM Test MSE: 1.3223857879638672


In [40]:
# Second LSTM configuration with different hyperparameters
hidden_size_2 = 128
num_layers_2 = 1

lstm_model_2 = LSTMModel(input_size, hidden_size_2, num_layers_2, output_size)

In [41]:
# Train second LSTM configuration

criterion = nn.MSELoss()

optimizer_2 = torch.optim.Adam(
    lstm_model_2.parameters(), lr=learning_rate, weight_decay=weight_decay
)

lstm2_losses = []

for epoch in range(epochs):

    lstm_model_2.train()

    optimizer_2.zero_grad()

    predictions = lstm_model_2(X_train_tensor)

    loss = criterion(predictions, y_train_tensor)

    loss.backward()

    optimizer_2.step()

    lstm2_losses.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

In [42]:
# Evaluate second LSTM configuration

lstm_model_2.eval()

with torch.no_grad():

    predictions = lstm_model_2(X_test_tensor)


# Convert predictions and actual values back to original scale

predictions_original = target_scaler.inverse_transform(predictions.numpy())

y_test_original = target_scaler.inverse_transform(y_test_tensor.numpy())


# Calculate MSE on original price scale

lstm_mse_2 = mean_squared_error(y_test_original, predictions_original)

print("LSTM Configuration 2 Test MSE:", lstm_mse_2)

LSTM Configuration 2 Test MSE: 2.831624984741211


LSTM Hyperparameter Comparison

Two LSTM configurations were tested with different hidden sizes and numbers of layers. The first model (hidden size = 64, 2 layers) achieved a Test MSE of 1.3224, while the second model (hidden size = 128, 1 layer) achieved a Test MSE of 2.8316. The first configuration performed better, suggesting that using an additional LSTM layer helped the model capture sequential patterns more effectively than simply increasing the hidden size.

In [43]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(GRUModel, self).__init__()

        # GRU is used because it can capture sequential dependencies
        # while using fewer gates than an LSTM, making it computationally efficient.
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        # Fully connected layer converts the learned sequence representation
        # into the final regression output.
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        # Pass input sequence through GRU layers
        out, _ = self.gru(x)

        # Use the final time step output for prediction
        out = self.fc(out[:, -1, :])

        return out

In [44]:
gru_model = GRUModel(input_size=input_size, hidden_size=50, num_layers=1, output_size=1)

In [45]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    gru_model.parameters(), lr=0.001, weight_decay=weight_decay
)

num_epochs = 50

gru_losses = []

for epoch in range(num_epochs):

    gru_model.train()

    optimizer.zero_grad()

    outputs = gru_model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    loss.backward()

    optimizer.step()

    gru_losses.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 10/50, Loss: 0.0915
Epoch 20/50, Loss: 0.0575
Epoch 30/50, Loss: 0.0596
Epoch 40/50, Loss: 0.0573
Epoch 50/50, Loss: 0.0577


In [46]:
gru_model.eval()

with torch.no_grad():

    predictions = gru_model(X_test_tensor)


# Convert predictions and true values back to original scale

predictions_original = target_scaler.inverse_transform(predictions.numpy())

y_test_original = target_scaler.inverse_transform(y_test_tensor.numpy())


# Calculate MSE on original price scale

gru_mse = mean_squared_error(y_test_original, predictions_original)

print("GRU Test MSE:", gru_mse)

GRU Test MSE: 1.3406482934951782


In [47]:
# Second GRU configuration with different hyperparameters

gru_model_2 = GRUModel(
    input_size=input_size, hidden_size=100, num_layers=2, output_size=1
)

In [48]:
criterion = nn.MSELoss()

optimizer_2 = torch.optim.Adam(
    gru_model_2.parameters(), lr=0.001, weight_decay=weight_decay
)

gru2_losses = []

for epoch in range(50):

    gru_model_2.train()

    optimizer_2.zero_grad()

    predictions = gru_model_2(X_train_tensor)

    loss = criterion(predictions, y_train_tensor)

    loss.backward()

    optimizer_2.step()

    gru2_losses.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50, Loss: {loss.item():.4f}")

Epoch 10/50, Loss: 0.0661
Epoch 20/50, Loss: 0.0603
Epoch 30/50, Loss: 0.0570
Epoch 40/50, Loss: 0.0569
Epoch 50/50, Loss: 0.0568


In [49]:
gru_model_2.eval()

with torch.no_grad():

    predictions = gru_model_2(X_test_tensor)


# Convert predictions and true values back to original scale

predictions_original = target_scaler.inverse_transform(predictions.numpy())

y_test_original = target_scaler.inverse_transform(y_test_tensor.numpy())


# Calculate MSE on original housing price scale

gru_mse_2 = mean_squared_error(y_test_original, predictions_original)

print("GRU Configuration 2 Test MSE:", gru_mse_2)

GRU Configuration 2 Test MSE: 1.3223400115966797


GRU Hyperparameter Comparison

Two GRU configurations were tested using different hidden sizes and numbers of layers. The first model (hidden size = 50, 1 layer) achieved a Test MSE of 1.3406, while the second model (hidden size = 100, 2 layers) achieved a lower Test MSE of 1.3223. The second configuration performed slightly better, suggesting that increasing both the hidden size and number of GRU layers improved the model's ability to learn sequential patterns from the housing data.

Effect of Removing Regularisation

a. Has performance gotten worse in both models?

The removal of regularisation did not result in the same performance change for both models. The impact depends on the model architecture and the dataset. Without regularisation, models may achieve better performance on the training data but can become more prone to overfitting. Regularisation helps improve generalisation by ensuring that the model performs well on unseen data.

b. What is the importance of regularisation for optimising the efficiency of models?

Regularisation is important because it reduces overfitting and improves a model's ability to generalise to new data. L2 regularisation, implemented through weight decay, limits excessively large model weights and encourages simpler, more stable models. This leads to more reliable predictions and improved performance when the model is evaluated on unseen data.